# Orchestration

LLMs have a training cutoff, so they may not know current APIs, updated documentation, private data, or the tools available in a system. Without this context, they can generate answers that sound plausible but are outdated or incorrect.

Orchestration solves this by connecting the model to the right context at runtime: documentation, retrieved data, and executable tools.

In this module, Kestra is used to implement three orchestration patterns: **AI Copilot** (documentation as context), **RAG** (data as context), and **AI Agents** (tools as context).


# Docker Compose environment


Everything is defined in a single `docker-compose.yml` at the root of the studies folder. The notebook, Elasticsearch, pgvector, and Kestra share the `llm-net` network, so they can communicate using their container names. Kestra's own PostgreSQL database is isolated in `kestra-net`, keeping its internal state separate from the application environment.

Secrets are passed to Kestra through environment variables prefixed with `SECRET_` and accessed in flows with `{{ secret('KEY_NAME') }}`. Regular variables use `${VAR}`. The full Kestra configuration is also passed through `KESTRA_CONFIGURATION`, avoiding the need for a separate config file.

The flows use Gemini as the main model provider, OpenAI in the RAG examples, and Tavily for web search in agent workflows. After import through the Kestra API, the flows appear in the `zoomcamp` namespace and can be executed from the UI or API.


The complete `docker-compose.yml`:

```yaml
# ==============================================================================
# LLM Zoomcamp study stack
# ==============================================================================
# One compose file for all local services used across the course modules.
# Each container has a single responsibility. Two Docker networks isolate
# Kestra's internal state from the general study network.
# ==============================================================================

services:
  # ----------------------------------------------------------------------------
  # notebook (container name: `studies`) — Jupyter Lab with GPU access.
  # Playground for notebooks, agent code, and experiments. Bind-mounts the
  # whole studies directory at /workspace, so files always live on the host
  # and survive container recreation.
  # ----------------------------------------------------------------------------
  notebook:
    build: .
    container_name: studies
    init: true
    restart: unless-stopped
    ports:
      - "9811:8888"                # Host 9811 -> Jupyter 8888 inside container
    volumes:
      - .:/workspace               # Bind mount: notebooks stay on the host
    working_dir: /workspace
    environment:
      # Connection info the notebook reads via os.getenv(...) at runtime.
      # Hostnames like "elasticsearch" and "pgvector" resolve on llm-net via
      # Docker's internal DNS (service name = hostname).
      JUPYTER_TOKEN: "${JUPYTER_TOKEN}"
      ELASTIC_HOST: "http://elasticsearch:9200"
      OPENAI_API_KEY: "${OPENAI_API_KEY}"
      POSTGRES_HOST: "pgvector"
      POSTGRES_PORT: "5432"
      POSTGRES_USER: "user"
      POSTGRES_PASSWORD: "pswd"
      POSTGRES_DB: "faq"
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia       # GPU access (embeddings, local LLM)
              count: all
              capabilities: [gpu]
    depends_on:
      - elasticsearch
      - pgvector
    networks:
      - llm-net

  # ----------------------------------------------------------------------------
  # elasticsearch — one tool for text search, vector search, and hybrid.
  # Uses the `dense_vector` field type (available since 7.14) for kNN search,
  # combined with BM25 for keyword search. This is the primary retrieval
  # engine for the final Obsidian agent project.
  # ----------------------------------------------------------------------------
  elasticsearch:
    image: docker.elastic.co/elasticsearch/elasticsearch:8.4.3
    container_name: elasticsearch
    restart: unless-stopped
    mem_limit: 8g
    environment:
      ES_JAVA_OPTS: "-Xms4g -Xmx4g"       # JVM heap: 4 GB min and max
      discovery.type: single-node
      xpack.security.enabled: "false"     # No auth (local dev only)
    ports:
      - "9200:9200"
    volumes:
      - es_data:/usr/share/elasticsearch/data
    networks:
      - llm-net

  # ----------------------------------------------------------------------------
  # pgvector — Postgres 17 with the `vector` extension enabled.
  # Kept for module 2 study (the SQL flavor of vector search). Can be removed
  # later if the final project settles on Elasticsearch alone.
  # ----------------------------------------------------------------------------
  pgvector:
    image: pgvector/pgvector:pg18
    container_name: pgvector
    restart: unless-stopped
    environment:
      POSTGRES_DB: faq
      POSTGRES_USER: user
      POSTGRES_PASSWORD: pswd
    ports:
      - "5432:5432"
    volumes:
      - pgvector_data:/var/lib/postgresql   # pg18+ mount convention (parent
                                            # dir, not /data subdir)
    networks:
      - llm-net

  # ----------------------------------------------------------------------------
  # kestra_postgres — dedicated Postgres for Kestra's internal state ONLY.
  #
  # ISOLATED: sits on `kestra-net` and NOT on `llm-net`. The `notebook`
  # container physically cannot reach it. Verify with:
  #   docker compose exec studies getent hosts kestra_postgres
  # (should return nothing).
  #
  # No ports published: only reachable inside kestra-net.
  # ----------------------------------------------------------------------------
  kestra_postgres:
    image: postgres:18
    container_name: kestra_postgres
    restart: unless-stopped
    environment:
      POSTGRES_DB: kestra
      POSTGRES_USER: kestra
      POSTGRES_PASSWORD: k3str4
    volumes:
      - kestra_postgres_data:/var/lib/postgresql   # pg18+ mount convention
    networks:
      - kestra-net

  # ----------------------------------------------------------------------------
  # kestra — workflow orchestrator with native AI plugins (RAG, Agents,
  # Copilot). Dual-network member:
  #   - kestra-net: to talk to its private Postgres backend.
  #   - llm-net:    to be reachable from the notebook (Python client can
  #                 call http://kestra:8080 from inside `studies`).
  #
  # Secrets: any env var prefixed `SECRET_*` becomes a Kestra secret. Inside
  # flows, reference them as {{ secret('KEY_NAME') }} (no SECRET_ prefix,
  # Kestra strips it). Values are base64-encoded; Kestra decodes at runtime.
  #
  # KESTRA_CONFIGURATION is a special env var containing the entire Kestra
  # config as a multi-line YAML string. Instead of mounting a config file,
  # Kestra reads its config from this variable on startup.
  # ----------------------------------------------------------------------------
  kestra:
    image: kestra/kestra:v1.3.21
    container_name: kestra
    restart: unless-stopped
    user: "root"
    command: server standalone
    volumes:
      - kestra_data:/app/storage
      - /var/run/docker.sock:/var/run/docker.sock  # Lets Kestra spawn tasks
                                                   # as Docker containers
      - /tmp/kestra-wd:/tmp/kestra-wd              # Task working directory
    environment:
      # Kestra secret store (base64-encoded).
      SECRET_GEMINI_API_KEY: ${SECRET_GEMINI_API_KEY}
      SECRET_TAVILY_API_KEY: ${SECRET_TAVILY_API_KEY}
      SECRET_OPENAI_API_KEY: ${SECRET_OPENAI_API_KEY}
      # Full Kestra config, injected as a multi-line YAML string.
      KESTRA_CONFIGURATION: |
        datasources:
          postgres:
            url: jdbc:postgresql://kestra_postgres:5432/kestra
            driverClassName: org.postgresql.Driver
            username: kestra
            password: k3str4
        kestra:
          server:
            basicAuth:
              username: "admin@kestra.io"
              password: Admin1234!
          repository:
            type: postgres
          storage:
            type: local
            local:
              basePath: "/app/storage"
          queue:
            type: postgres
          tasks:
            tmpDir:
              path: /tmp/kestra-wd/tmp
          url: http://localhost:8080/
          ai:
            # AI Copilot config (the sparkle button inside the Kestra UI).
            # Needs the RAW (non-base64) Gemini key here.
            type: gemini
            gemini:
              model-name: gemini-2.5-flash
              api-key: ${GEMINI_API_KEY}
    ports:
      - "8080:8080"                 # Web UI + REST API
      - "8081:8081"                 # Internal health check endpoint
    depends_on:
      - kestra_postgres
    networks:
      - kestra-net
      - llm-net

# ==============================================================================
# Networks
# ==============================================================================
# llm-net    — shared network. Every service the notebook needs to reach
#              lives here (elasticsearch, pgvector, kestra).
# kestra-net — private network for Kestra and its Postgres. Not accessible
#              from the notebook container. Enforces isolation of the
#              orchestrator's internal state.
# ==============================================================================
networks:
  llm-net:
    driver: bridge
  kestra-net:
    driver: bridge

# ==============================================================================
# Named volumes (persist across container recreation and restarts)
# ==============================================================================
# es_data              — Elasticsearch indices
# pgvector_data        — Postgres data (databases, embeddings)
# kestra_postgres_data — Kestra internal state (flows, executions, queue)
# kestra_data          — Kestra file storage (flow YAMLs, task outputs)
# ==============================================================================
volumes:
  es_data:
  pgvector_data:
  kestra_postgres_data:
  kestra_data:
```

# Retrieval-Augmented Generation in Kestra

RAG is useful when the information needed to answer a question is not reliably inside the model: it may be recent documentation, internal data, or a live external source.

Instead of relying only on the model's training data, the flow retrieves relevant content and passes it as context before generating the answer. This grounds the response on an actual source.

Kestra provides this through the `io.kestra.plugin.ai.rag.*` plugins. The main task is `rag.ChatCompletion`, which handles the retrieve → augment → generate pipeline internally. The main decision is where the context comes from:

- an embedding store, for previously indexed documents;
- a live retriever, such as web search;
- or both, in a hybrid setup.

The examples below show the progression from no context, to static RAG, to live RAG.

**Baseline: LLM without RAG**

This first flow only sends a question to Gemini. Since no release documentation is provided, the model has to rely on its training data and may return outdated or invented features.

```yaml
id: 1_chat_without_rag
namespace: zoomcamp

tasks:
  - id: chat_without_rag
    type: io.kestra.plugin.ai.completion.ChatCompletion
    provider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-2.5-flash
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    messages:
      - type: USER
        content: |
          Which features were released in Kestra 1.1?
          Please list at least 5 major features with brief descriptions.

  - id: log_results
    type: io.kestra.plugin.core.log.Log
    message: "{{ outputs.chat_without_rag.textOutput }}"
```

`completion.ChatCompletion` is the plain LLM task. It receives a prompt and returns a response, but it has no retrieval step, tool use, or external context.

The second task only logs the output from the first one. Kestra accesses outputs with expressions such as:

```yaml
{{ outputs.chat_without_rag.textOutput }}
```

This is the basic way one task passes data to another in a flow.

**Static RAG: release notes indexed in KV Store**

The second flow first ingests the Kestra 1.1 release note, then queries it through `rag.ChatCompletion`.

```yaml
id: 2_chat_with_rag
namespace: zoomcamp

tasks:
  - id: ingest_release_notes
    type: io.kestra.plugin.ai.rag.IngestDocument
    provider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-embedding-001
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddings:
      type: io.kestra.plugin.ai.embeddings.KestraKVStore
    drop: true
    fromExternalURLs:
      - https://raw.githubusercontent.com/kestra-io/docs/refs/heads/main/src/contents/blogs/release-1-1/index.md

  - id: chat_with_rag
    type: io.kestra.plugin.ai.rag.ChatCompletion
    chatProvider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-2.5-flash
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddingProvider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-embedding-001
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddings:
      type: io.kestra.plugin.ai.embeddings.KestraKVStore
    systemMessage: |
      Answer using the retrieved Kestra documentation.
      If the answer is not in the context, say so.
    prompt: |
      Which features were released in Kestra 1.1?
      Please list at least 5 major features with brief descriptions.

  - id: log_results
    type: io.kestra.plugin.core.log.Log
    message: "{{ outputs.chat_with_rag.textOutput }}"
```

`IngestDocument` downloads the document, splits it into chunks, converts the chunks into embeddings, and stores them in Kestra KV Store.

Then, `rag.ChatCompletion` embeds the question, retrieves the most similar chunks, adds them to the prompt, and sends the final context to the chat model.

The embedding model used during ingestion and query must be the same, because both document chunks and the prompt need to be represented in the same vector space.

The KV Store is enough for the course examples, but it is not the best choice for a larger knowledge base. For production-scale data, the same flow can use pgvector, Elasticsearch, Qdrant, or another vector database by changing the `embeddings` block.

**Dynamic RAG: live web search**

The third flow does not index documents beforehand. Instead, it retrieves fresh information from the web at each execution.

```yaml
id: 3_rag_with_websearch
namespace: zoomcamp

tasks:
  - id: chat_with_rag_and_websearch
    type: io.kestra.plugin.ai.rag.ChatCompletion
    chatProvider:
      type: io.kestra.plugin.ai.provider.OpenAI
      modelName: gpt-5-mini
      apiKey: "{{ secret('OPENAI_API_KEY') }}"
    contentRetrievers:
      - type: io.kestra.plugin.ai.retriever.TavilyWebSearch
        apiKey: "{{ secret('TAVILY_API_KEY') }}"
    systemMessage: |
      You are a helpful assistant that answers questions about Kestra.
    prompt: What is the latest release of Kestra?
```

The main difference from the previous flow is that `embeddings` and `embeddingProvider` are replaced by `contentRetrievers`.

Tavily performs the search and returns relevant text snippets directly. Kestra then injects those snippets into the context sent to the model.

This is useful when the source changes frequently or when it does not make sense to maintain a local index. The trade-off is that each execution requires an external search request, so the result may vary over time and the flow is less reproducible.

**changes between the flows**

| Flow        | Retrieval source | Ingest step | Best use case                       |
| ----------- | ---------------- | ----------: | ----------------------------------- |
| No RAG      | None             |          No | Baseline comparison                 |
| Static RAG  | Embedding store  |         Yes | Internal or stable documents        |
| Dynamic RAG | Web search       |          No | Recent or fast-changing information |

The important point is that the core RAG task stays almost the same. `rag.ChatCompletion` always receives a prompt and a chat provider. What changes is only the retrieval layer: `embeddings` for indexed content or `contentRetrievers` for live sources.

This makes it easy to switch between static, dynamic, or hybrid RAG without redesigning the whole workflow.


# Homework

## Question 1: Context Engineering

Try the following experiment:

1. Open ChatGPT in a private browser window: https://chatgpt.com
2. Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"
3. Then, use Kestra's AI Copilot with the same prompt

After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?

- AI Copilot uses a more powerful model
- **AI Copilot has access to current Kestra plugin documentation**
- AI Copilot uses more tokens
- AI Copilot has internet access



## Question 2: RAG vs No RAG

Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml` in the Kestra UI. Read the execution logs for each.

The non-RAG response about Kestra 1.1 features is best described as:

- Accurate and specific, matching the actual release notes
- **Vague, generic, or fabricated — the model guesses from training data**
- Empty — the model refuses to answer without context
- Identical to the RAG version



## Question 3: Token usage - short summary

Run `4_simple_agent.yaml` with `summary_length = short` (leave the other inputs as defaults).

Open the execution logs and find the token usage logged by the `log_token_usage` task.

What is the approximate **output** token count for `multilingual_agent`?

- 5-15 tokens
- **60-100 tokens**
- 200-400 tokens
- 500+ tokens


2026-07-07 18:12:37.749📊 Token Usage Summary:

Multilingual Agent:

- Input tokens: 282
- **Output tokens: 86**
- Total tokens: 368

English Brevity Agent:

- Input tokens: 101
- Output tokens: 45
- Total tokens: 146

💡 Tip: Monitor token usage to understand costs and optimize prompts!


## Question 4: Token usage - long summary

Run `4_simple_agent.yaml` again with `summary_length = long`.

Compare the `multilingual_agent` output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?

- About the same (within 20%)
- **2-5x more** **(179/86 ≈ 2.08x)**
- 10-20x more
- 50x more


2026-07-07 18:14:32.809 📊 Token Usage Summary:

Multilingual Agent:

- Input tokens: 282
- **Output tokens: 179**
- Total tokens: 461

English Brevity Agent:

- Input tokens: 194
- Output tokens: 47
- Total tokens: 241

💡 Tip: Monitor token usage to understand costs and optimize prompts!


## Question 5: Modifying a flow

Open `4_simple_agent.yaml` in the Kestra flow editor. Find the `english_brevity` task and change its prompt from asking for exactly **1 sentence** to asking for exactly **3 sentences**.

Save the flow, then run it with `summary_length = long`.

Compare the `english_brevity` output token count to the original 1-sentence version (also with `summary_length = long`). How do they compare?

- About the same (within 20%)
- **2-4x more (85 / 47 ≈ 1.81x)**
- 5-10x more
- 10x+ more


2026-07-07 18:24:14.608📊 Token Usage Summary:

Multilingual Agent:

- Input tokens: 282
- Output tokens: 184
- Total tokens: 466

English Brevity Agent:

- Input tokens: 199
- **Output tokens: 85**
- Total tokens: 284

💡 Tip: Monitor token usage to understand costs and optimize prompts!


## Question 6: Best Practices

Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- **Use traditional task-based workflows for predictability and auditability**
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data

